# Train the sleep stage models in Google Colab (free)

This notebook does on Google's computers what `scripts/train_models.py` does on a laptop,
but with **all** of BIDSleep (47 people) instead of the 3-night sample, and with a free graphics card.

**Cost: nothing.** Everything here uses the free Colab tier and, optionally, your free Google Drive space.
If Colab ever offers "Colab Pro", "compute units" or a paid GPU, click **No thanks**: none of it is needed.

## Before you start (click by click)

1. Go to **https://colab.research.google.com** and sign in with a Google account.
2. Click **File → Open notebook**, choose the **GitHub** tab, paste
   `https://github.com/ademchuk2024/group3-sleep-monitor`, press Enter, and pick `notebooks/train_models.ipynb`.
3. Click **Runtime → Change runtime type**, choose **T4 GPU**, click **Save**.
4. Run each cell from top to bottom: click inside it and press **Shift + Enter** (or click ▶ on its left).
   Wait for a cell to finish (its ▶ stops spinning) before starting the next.

**How long:** about 1–2 hours for downloads and preparation, then about 1–2 hours of training.
A free session lasts several hours but stops if the browser tab is idle for long, so keep the tab open.
Step 4 can save the prepared data to your Google Drive, so after a disconnect you can skip the downloads.

**Privacy:** these are public research datasets, used under their licences. They stay on Colab's disk
and in *your own* Drive. Never upload them to GitHub or share them.

## 1. Check that a graphics card is attached
You should see a table naming a **Tesla T4**. If it says the command was not found, redo step 3 above.

In [ ]:
!nvidia-smi

## 2. Get the project code

In [ ]:
!git clone https://github.com/ademchuk2024/group3-sleep-monitor.git
%cd group3-sleep-monitor
!git log -1 --oneline

## 3. Install the packages

PyTorch is already in Colab. This installs the exact versions the project was tested with.
If a box appears saying **"Restart session"**, click it, then run the small cell below this one (it returns to the project folder) and carry on.

In [ ]:
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
%cd /content/group3-sleep-monitor

## 4. (Optional) Keep the prepared data in your Google Drive

Run this to connect your Drive. Google will ask you to allow access: click **Connect to Google Drive**, pick your account, click **Allow**.
The prepared data (about 0.5 GB) is then saved to `My Drive/sleep-monitor-features`, so a later session can skip steps 5 and 6.
Skip this cell if you would rather not use Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FEATURES = '/content/drive/MyDrive/sleep-monitor-features'
import os
if os.path.isdir(DRIVE_FEATURES):
    !mkdir -p data && cp -r "$DRIVE_FEATURES" data/features
    print('Found prepared data in Drive:', os.listdir('data/features'), '-> you can skip steps 5 and 6.')

## 5. Download the datasets (about 36 GB, the longest step)

Walch (2.4 GB), Newcastle (5.4 GB) and the full BIDSleep (about 28 GB). Each file is checked after downloading.
DREAMT is **not** downloaded: it needs approved access.

In [ ]:
!python scripts/download_data.py walch
!python scripts/download_data.py newcastle
!python scripts/download_data.py bidsleep

## 6. Prepare the data (signal processing, the same code as the app)

`default` uses 1-second frames. `frame2` uses 2-second frames, for the frame-length comparison.

In [ ]:
!python scripts/build_features.py --tag default
!python scripts/build_features.py --tag frame2 --frame-seconds 2 --overlap 0.75

Save the prepared data to Drive (only if you ran step 4):

In [ ]:
!mkdir -p "$DRIVE_FEATURES" && cp -r data/features/* "$DRIVE_FEATURES"/

## 7. Quick check that everything works
All tests should pass.

In [ ]:
!python -m pytest -q tests/test_models.py tests/test_evaluate.py tests/test_features.py

## 8. Compare the choices, on the VALIDATION people

Each line trains one model and scores it on the validation people (never on the test people).

In [ ]:
T = 'python scripts/train_models.py'
# The four models with the default settings
!{T} --kinds forest dnn cnn sequence
# Same model, other random starts: shows how much scores move by chance
!{T} --kinds sequence --seed 1 --label sequence-seed1
!{T} --kinds sequence --seed 2 --label sequence-seed2
# xyz versus magnitude
!{T} --kinds forest sequence --version magnitude
# Ways of dealing with the noisier GENEActiv
!{T} --kinds forest sequence --norm absolute
!{T} --kinds forest sequence --norm relative
# 1 s versus 2 s frames
!{T} --kinds forest sequence --tag frame2
# Class imbalance: without the balancing weights
!{T} --kinds forest dnn cnn sequence --unbalanced
# Heart rate (only Walch and BIDSleep have it)
!{T} --kinds sequence --datasets walch bidsleep
!{T} --kinds sequence --datasets walch bidsleep --heart-rate

In [ ]:
!python scripts/summarise_results.py --on validation

## 9. Final scores on the TEST people (run once)

Set the choices below to what won in step 8 (macro F1, and differences bigger than the seed-to-seed spread).
The values filled in are the ones that won on the laptop run. Then run the cell **once**.
Re-running it after looking at test scores and changing settings would make the test scores too optimistic.

In [ ]:
TAG, VERSION, NORM = 'default', 'magnitude', 'common_floor'
!python scripts/train_models.py --kinds forest dnn cnn sequence --tag {TAG} --version {VERSION} --norm {NORM} --score-on test --save
!python scripts/train_models.py --kinds sequence --tag {TAG} --version {VERSION} --norm {NORM} --datasets walch bidsleep --score-on test
!python scripts/train_models.py --kinds sequence --tag {TAG} --version {VERSION} --norm {NORM} --datasets walch bidsleep --heart-rate --score-on test
!python scripts/summarise_results.py --on test

## 10. Make the app's model file and download the results

This exports the best model to ONNX (checked against the original), then packs the results into one zip.
Your browser downloads `colab_results.zip`. Put its contents into the project folder on your laptop
(`results/` and `models/stage_model/`) and commit them. They hold scores and the network only, no recordings.

In [ ]:
import glob
best = glob.glob(f'data/models/sequence-{TAG}-{VERSION}-{NORM}-balanced-test.pt')[0]
!python scripts/export_model.py {best}
!zip -qr colab_results.zip results models/stage_model
from google.colab import files
files.download('colab_results.zip')